In [1]:
import pandas as pd
print (pd.__version__)

3.1.0


# Predicting Premier League Home Wins
## 1. Setup

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd

DATA_DIR = Path("data") #stores in data
SEASONS = ["1516", "1617", "1718", "1819", "1920",
           "2021", "2122", "2223", "2324", "2425"] #which seasons ill be working with

## 2. Data loading

In [ ]:
DATA_DIR.mkdir(exist_ok=True)
#download data and store as csv file
for season in SEASONS:
    file_path = DATA_DIR / f"E0_{season}.csv"
    if file_path.exists():
        print("Already have", file_path)
    else:
        url = f"https://football-data.co.uk/mmz4281/{season}/E0.csv"
        urllib.request.urlretrieve(url, file_path)
        print("Downloaded", file_path)

Downloaded data\E0_1516.csv
Downloaded data\E0_1617.csv
Downloaded data\E0_1718.csv
Downloaded data\E0_1819.csv
Downloaded data\E0_1920.csv
Downloaded data\E0_2021.csv
Downloaded data\E0_2122.csv
Downloaded data\E0_2223.csv
Downloaded data\E0_2324.csv
Downloaded data\E0_2425.csv


In [ ]:
frames = []
for season in SEASONS:
    df_season = pd.read_csv(DATA_DIR / f"E0_{season}.csv") #read csv files
    df_season["Season"] = season
    frames.append(df_season)

matches = pd.concat(frames, ignore_index=True) #combine
matches.shape 

C:\Users\adam1_4sss501\AppData\Local\Temp\ipykernel_21708\2881986675.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_season["Season"] = season


(3800, 154)

## 3. Data audit

In [ ]:
missing_pct = matches.isna().mean() * 100 #calculate missing values
missing_pct.sort_values(ascending=False).head(20)

1XBD       90.236842
1XBH       90.236842
1XBA       90.236842
BFE>2.5    90.078947
BFE<2.5    90.078947
BFA        90.026316
BFH        90.026316
BFD        90.026316
1XBCD      90.000000
1XBCH      90.000000
BFECH      90.000000
1XBCA      90.000000
BFEAHA     90.000000
BFCH       90.000000
BFCD       90.000000
BFCA       90.000000
BFECAHA    90.000000
BFECAHH    90.000000
BFEH       90.000000
BFEAHH     90.000000
dtype: float64

In [ ]:
key_cols = ["Date", "HomeTeam", "AwayTeam", "FTR",
            "FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC",
            "B365H", "B365D", "B365A",
            "AvgH", "AvgD", "AvgA", "BbAvH", "BbAvD", "BbAvA"]
missing_pct[key_cols] #show missing data percent for my columns

Date         0.0
HomeTeam     0.0
AwayTeam     0.0
FTR          0.0
FTHG         0.0
FTAG         0.0
HS           0.0
AS           0.0
HST          0.0
AST          0.0
HC           0.0
AC           0.0
B365H        0.0
B365D        0.0
B365A        0.0
AvgH        40.0
AvgD        40.0
AvgA        40.0
BbAvH       60.0
BbAvD       60.0
BbAvA       60.0
dtype: float64

In [19]:
keep_cols = ["Season", "Date", "HomeTeam", "AwayTeam", "FTR",
             "FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC",
             "B365H", "B365D", "B365A"] #removed useless columns
df = matches[keep_cols].copy()
df.head() #first five rows
#df["HomeTeam"].nunique()

,Season,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A
0,1516,08/08/2015,Bournemouth,Aston Villa,A,0,1,11,7,2,3,6,3,2.00,3.6,4.00
1,1516,08/08/2015,Chelsea,Swansea,D,2,2,11,18,3,10,4,8,1.36,5.0,11.00
2,1516,08/08/2015,Everton,Watford,D,2,2,10,11,5,5,8,2,1.70,3.9,5.50
3,1516,08/08/2015,Leicester,Sunderland,H,4,2,19,10,8,5,6,3,1.95,3.5,4.33
4,1516,08/08/2015,Man United,Tottenham,H,1,0,9,9,1,4,1,2,1.65,4.0,6.00


In [20]:
df.dtypes

Season          str
Date            str
HomeTeam        str
AwayTeam        str
FTR             str
FTHG          int64
FTAG          int64
HS            int64
AS            int64
HST           int64
AST           int64
HC            int64
AC            int64
B365H       float64
B365D       float64
B365A       float64
dtype: object

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed") #text to real dates and time format fix
df = df.sort_values("Date").reset_index(drop=True) #order match 
df.groupby("Season")["Date"].agg(["min", "max"]) #first vs last

,min,max
Season,,
1516,2015-08-08,2016-05-17
1617,2016-08-13,2017-05-21
1718,2017-08-11,2018-05-13
1819,2018-08-10,2019-05-12
1920,2019-08-09,2020-07-26
2021,2020-09-12,2021-05-23
2122,2021-08-13,2022-05-22
2223,2022-08-05,2023-05-28
2324,2023-08-11,2024-05-19
